# Creatubg your own dataset

In [ ]:
import requests

url = "https://api.github.com/repos/huggingface/datasets/issues?page=1&per_page=1"

response = requests.get(url)

In [ ]:
response.status_code 

In [ ]:
response.json()

## GitHub Docs

`https://docs.github.com/en/authentication/keeping-your-account-and-data-secure/managing-your-personal-access-tokens`

In [ ]:
GITHUB_TOKEN = "YOUR HF TOKEN"
headers = {"Authorization":f"token {GITHUB_TOKEN}"}

In [ ]:
import os
import time
import requests
import pandas as pd

from pathlib import Path
from tqdm import tqdm
from dotenv import load_dotenv


load_dotenv()

# GITHUB_TOKEN = os.getenv("GITHUB_TOKEN")

headers = {
    "Authorization": f"Bearer {GITHUB_TOKEN}",
    "Content-Type": "application/json",
}


def fetch_issues(
    owner="huggingface",
    repo="datasets",
    num_issues=10_000,
    issues_path=Path("."),
):

    issues_path.mkdir(parents=True, exist_ok=True)

    url = "https://api.github.com/graphql"

    query = """
    query($owner: String!, $repo: String!, $first: Int!, $after: String) {
      repository(owner: $owner, name: $repo) {
        issues(
          first: $first,
          after: $after,
          orderBy: {field: CREATED_AT, direction: DESC}
        ) {
          nodes {
            id
            number
            title
            body
            state
            createdAt
            updatedAt
            closedAt
            url

            author {
              login
            }

            labels(first: 20) {
              nodes {
                name
              }
            }

            comments(first: 20) {
              nodes {
                body
                createdAt

                author {
                  login
                }
              }
            }
          }

          pageInfo {
            hasNextPage
            endCursor
          }
        }
      }
    }
    """

    all_issues = []
    cursor = None

    pbar = tqdm(total=num_issues)

    while len(all_issues) < num_issues:

        variables = {
            "owner": owner,
            "repo": repo,
            "first": min(100, num_issues - len(all_issues)),
            "after": cursor,
        }

        response = requests.post(
            url,
            headers=headers,
            json={
                "query": query,
                "variables": variables,
            },
        )

        if response.status_code != 200:
            print("Status:", response.status_code)
            print(response.text)
            break

        data = response.json()

        if "errors" in data:
            print("GraphQL errors:")
            print(data["errors"])
            break

        repository = data["data"]["repository"]

        if repository is None:
            print("Repository not found.")
            break

        issues_data = repository["issues"]

        issues = issues_data["nodes"]

        for issue in issues:
            all_issues.append(issue)

            pbar.update(1)

            if len(all_issues) >= num_issues:
                break

        page_info = issues_data["pageInfo"]

        if not page_info["hasNextPage"]:
            break

        cursor = page_info["endCursor"]

    pbar.close()

    # Convert nested data into a DataFrame
    df = pd.DataFrame.from_records(all_issues)

    output_path = issues_path / f"{repo}-issues.jsonl"

    df.to_json(
        output_path,
        orient="records",
        lines=True,
    )

    print()
    print(f"Downloaded {len(all_issues)} issues!")
    print(f"Dataset stored at: {output_path}")

In [ ]:
fetch_issues()

In [ ]:
from datasets import load_dataset

issues_dataset = load_dataset("json", data_files="datasets-issues.jsonl", split="train")
issues_dataset

## Cleaning up the data


In [ ]:
sample = issues_dataset.shuffle(seed=42).select(range(3))

sample

In [ ]:
# id data have pull_request column 
issues_dataset = issues_dataset.map(
    lambda x: {"is_pull_request": False if x["pull_request"] is None else True}
)

## Augmenting the dataset


In [ ]:
issue_number = 2792

url = f"https://api.github.com/repos/huggingface/datasets/issues/{issue_number}/comments"

response = requests.get(url, headers=headers)

response.json()

In [ ]:
def get_comments(issue_number):
    url = f"https://api.github.com/repos/huggingface/datasets/issues/{issue_number}/comments"
    response = requests.get(url, headers=headers)
    return [r["body"] for r in response.json()]


# Test our function works as expected
get_comments(2792)

In [ ]:
get_comments(2444)

In [ ]:
issues_with_comments_datasets = issues_dataset.map(
    lambda x: {"comments": get_comments(x["number"])}
)

In [ ]:
issues_with_comments_datasets

In [ ]:
# from huggingface_hub import notebook_login

# notebook_login()

In [ ]:
# from huggingface_hub import whoami

# whoami()

In [ ]:
issues_with_comments_datasets.push_to_hub("farid678/github-issues")